# 04 - Model Training: GSE96058 / SCAN-B

Train and evaluate classifiers on GSE96058 with combined features (17 total):
- 8 pathway features (7 pathways + ratio)
- 9 clinical features (encoded)

**Setup**: Stratified 5-fold cross-validation, StandardScaler within each fold.

**Expected results**: EN AUC=0.855, RF AUC=0.856, GB AUC=0.827

**Requires**: Downloaded GSE96058 expression data.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np

from src.data_loader import load_clinical_data, load_gse96058_expression
from src.preprocessing import zscore_normalize, encode_clinical_features, filter_outcome
from src.features import compute_pathway_scores, add_ratio_features, build_feature_matrix
from src.models import get_classifiers, run_cv_evaluation

## 1. Load and Prepare Data

In [ ]:
# Load clinical data
gse_clin = load_clinical_data('../data/clinical/01_gse96058_clinical.csv')
gse_clin = filter_outcome(gse_clin)

# Load and process expression data
gse_exp = load_gse96058_expression('../data/raw/GSE96058_gene_expression.csv')

# Align samples between expression and clinical
common_samples = list(set(gse_clin['sample_id']) & set(gse_exp['sample_id']))
print(f"Common samples: {len(common_samples)}")

gse_clin = gse_clin[gse_clin['sample_id'].isin(common_samples)].reset_index(drop=True)
gse_exp = gse_exp[gse_exp['sample_id'].isin(common_samples)].reset_index(drop=True)

# Sort both by sample_id to ensure alignment
gse_clin = gse_clin.sort_values('sample_id').reset_index(drop=True)
gse_exp = gse_exp.sort_values('sample_id').reset_index(drop=True)

## 2. Compute Features

In [ ]:
# Z-score normalize expression
print("Z-score normalizing expression data...")
gse_exp_norm = zscore_normalize(gse_exp)

# Compute pathway scores
print("Computing pathway scores...")
pathway_scores = compute_pathway_scores(gse_exp_norm)
pathway_scores = add_ratio_features(pathway_scores)

# Encode clinical features
print("Encoding clinical features...")
clinical_features = encode_clinical_features(gse_clin)

# Build combined feature matrix
X_combined = build_feature_matrix(pathway_scores, clinical_features)
y = gse_clin['high_risk'].values

print(f"\nCombined feature matrix: {X_combined.shape}")
print(f"Features: {X_combined.columns.tolist()}")

## 3. Cross-Validation (Combined Features)

In [ ]:
print("Running 5-fold stratified CV on combined features (17)...")
results = run_cv_evaluation(X_combined, y)

print("\n" + "=" * 60)
print("GSE96058 RESULTS (Combined, 17 Features)")
print("=" * 60)
for _, row in results.iterrows():
    print(f"  {row['Model']:20s}  AUC: {row['AUC']:.3f} +/- {row['AUC_SD']:.3f}  Acc: {row['Accuracy']:.3f}")

## 4. Append Results

In [ ]:
# Load existing TCGA results and append GSE96058
tcga_results = pd.read_csv('../results/03_model_performance.csv')

gse_results = results.copy()
gse_results.insert(0, 'Dataset', 'GSE96058')
gse_results = gse_results.rename(columns={'AUC_SD': 'Std'})

combined_results = pd.concat([tcga_results, gse_results], ignore_index=True)
combined_results.to_csv('../results/03_model_performance.csv', index=False)
print("\nUpdated results/03_model_performance.csv")
print(combined_results.to_string(index=False))